In [ ]:
"""
TASK 2: Exploratory Data Analysis (EDA) & Business Intelligence
ApexPlanet Software Pvt. Ltd. - Data Analytics Internship
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#e6edf3',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'text.color': '#e6edf3',
    'grid.color': '#21262d',
    'grid.linestyle': '--',
    'grid.alpha': 0.5,
    'font.family': 'DejaVu Sans',
})
ACCENT = ['#58a6ff','#3fb950','#f78166','#d2a8ff','#ffa657','#79c0ff','#56d364']

print("=" * 60)
print("TASK 2: Exploratory Data Analysis & Business Intelligence")
print("=" * 60)

df = pd.read_csv('/content/sample_data/cleaned_dataset.csv', parse_dates=['transaction_date'])
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns\n")

# ─── STEP 1: DESCRIPTIVE STATISTICS ─────────────────────────
print("--- STEP 1: DESCRIPTIVE STATISTICS ---")
print("\nNumerical Summary:")
print(df[['transaction_amount','account_balance','credit_score','emi_amount']].describe().round(2))

print("\nTransaction Direction Split:")
print(df['transaction_direction'].value_counts(normalize=True).mul(100).round(2).to_string())

print("\nTransaction Status Split:")
print(df['transaction_status'].value_counts(normalize=True).mul(100).round(2).to_string())

print("\nFraud Rate:", f"{df['is_fraud'].mean()*100:.3f}%")

# ─── FIGURE 1: UNIVARIATE DISTRIBUTIONS ─────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('TASK 2 — Univariate Analysis', fontsize=16, fontweight='bold', color='#58a6ff', y=1.01)

# 1a. Transaction amount (log scale)
ax = axes[0,0]
log_amt = np.log1p(df['transaction_amount'])
ax.hist(log_amt, bins=60, color=ACCENT[0], edgecolor='none', alpha=0.85)
ax.set_title('Transaction Amount (log scale)', color='#e6edf3')
ax.set_xlabel('log(1 + Amount ₹)')

# 1b. Account balance distribution
ax = axes[0,1]
ax.hist(np.log1p(df['account_balance']), bins=60, color=ACCENT[1], edgecolor='none', alpha=0.85)
ax.set_title('Account Balance (log scale)', color='#e6edf3')
ax.set_xlabel('log(1 + Balance ₹)')

# 1c. Credit Score
ax = axes[0,2]
ax.hist(df['credit_score'], bins=50, color=ACCENT[3], edgecolor='none', alpha=0.85)
ax.set_title('Credit Score Distribution', color='#e6edf3')
ax.set_xlabel('Credit Score')

# 1d. Account Type
ax = axes[1,0]
vc = df['account_type'].value_counts()
bars = ax.bar(vc.index, vc.values, color=ACCENT[:len(vc)], edgecolor='none')
ax.set_title('Account Types', color='#e6edf3')
ax.set_xlabel('Account Type')
ax.set_ylabel('Count')
for b in bars:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+500, f'{b.get_height():,.0f}',
            ha='center', va='bottom', fontsize=8, color='#e6edf3')

# 1e. Transaction Type
ax = axes[1,1]
vc2 = df['transaction_type'].value_counts()
ax.barh(vc2.index, vc2.values, color=ACCENT[0], edgecolor='none', alpha=0.85)
ax.set_title('Transaction Types', color='#e6edf3')
ax.set_xlabel('Count')

# 1f. Channel
ax = axes[1,2]
vc3 = df['channel'].value_counts()
wedges, texts, autos = ax.pie(vc3.values, labels=vc3.index, autopct='%1.1f%%',
    colors=ACCENT[:len(vc3)], startangle=90,
    wedgeprops={'edgecolor':'#0d1117','linewidth':1.5})
for t in texts+autos:
    t.set_color('#e6edf3')
    t.set_fontsize(9)
ax.set_title('Transaction Channels', color='#e6edf3')

plt.tight_layout()
plt.savefig('/content/sample_data/fig1_univariate.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.close()
print("[FIG 1] Saved: fig1_univariate.png")

# ─── STEP 2: SQL-STYLE BUSINESS QUESTIONS ────────────────────
print("\n--- STEP 2: SQL-STYLE BUSINESS QUESTIONS ---")

# Q1: Top 5 merchant categories by total transaction volume
q1 = (df[df['transaction_direction']=='Debit']
      .groupby('merchant_category')['transaction_amount']
      .sum().sort_values(ascending=False).head(5))
print("\nQ1 — Top 5 Merchant Categories by Revenue (Debit):")
print(q1.apply(lambda x: f"₹{x:,.0f}").to_string())

# Q2: Monthly transaction trend
q2 = df.groupby(['year','month']).agg(
    txn_count=('transaction_id','count'),
    total_volume=('transaction_amount','sum')
).reset_index()
q2['period'] = pd.to_datetime(dict(year=q2['year'], month=q2['month'], day=1))
q2 = q2.sort_values('period')
print("\nQ2 — Monthly Transaction Count (sample last 6):")
print(q2[['period','txn_count','total_volume']].tail(6).to_string(index=False))

# Q3: Fraud rate by state
q3 = df.groupby('state').agg(
    total=('is_fraud','count'),
    fraud=('is_fraud','sum')
).assign(fraud_rate=lambda x: x['fraud']/x['total']*100).sort_values('fraud_rate', ascending=False)
print("\nQ3 — Fraud Rate by State (%):")
print(q3[['fraud','total','fraud_rate']].round(3).to_string())

# Q4: Success rate by channel
q4 = df.groupby('channel').apply(
    lambda x: (x['transaction_status']=='Success').sum() / len(x) * 100
).round(2).sort_values(ascending=False)
print("\nQ4 — Transaction Success Rate by Channel (%):")
print(q4.to_string())

# Q5: Average transaction amount by account type and direction
q5 = df.groupby(['account_type','transaction_direction'])['transaction_amount'].mean().unstack().round(2)
print("\nQ5 — Avg Transaction Amount by Account Type & Direction (₹):")
print(q5.to_string())

# Q6: KYC compliance impact on fraud
q6 = df.groupby('kyc_status')['is_fraud'].agg(['mean','sum','count'])
q6['fraud_rate_%'] = (q6['mean']*100).round(3)
print("\nQ6 — Fraud by KYC Status:")
print(q6[['sum','count','fraud_rate_%']].to_string())

# Q7: Top 5 states by transaction volume
q7 = df.groupby('state')['transaction_amount'].sum().sort_values(ascending=False)
print("\nQ7 — Top States by Total Transaction Volume:")
print(q7.apply(lambda x: f"₹{x:,.0f}").to_string())

# ─── FIGURE 2: MULTIVARIATE ANALYSIS ─────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(20, 11))
fig.suptitle('TASK 2 — Multivariate Analysis & Correlation', fontsize=16,
             fontweight='bold', color='#58a6ff', y=1.01)

# 2a. Monthly trend
ax = axes[0,0]
ax.plot(q2['period'], q2['txn_count']/1000, color=ACCENT[0], linewidth=2.5, marker='o', markersize=3)
ax.fill_between(q2['period'], q2['txn_count']/1000, alpha=0.15, color=ACCENT[0])
ax.set_title('Monthly Transaction Count (K)', color='#e6edf3')
ax.set_xlabel('Month'); ax.set_ylabel('Transactions (K)')
ax.tick_params(axis='x', rotation=45)

# 2b. Fraud rate by state
ax = axes[0,1]
q3_sorted = q3.sort_values('fraud_rate')
colors_fraud = [ACCENT[2] if r > q3['fraud_rate'].mean() else ACCENT[1] for r in q3_sorted['fraud_rate']]
bars = ax.barh(q3_sorted.index, q3_sorted['fraud_rate'], color=colors_fraud, edgecolor='none')
ax.axvline(q3['fraud_rate'].mean(), color='#ffa657', linestyle='--', linewidth=1.5, label='Avg')
ax.set_title('Fraud Rate by State (%)', color='#e6edf3')
ax.set_xlabel('Fraud Rate (%)')
ax.legend(fontsize=8)

# 2c. Credit Score vs Amount (scatter sample)
ax = axes[0,2]
sample = df.sample(5000, random_state=42)
sc = ax.scatter(sample['credit_score'], np.log1p(sample['transaction_amount']),
                c=sample['is_fraud'], cmap='RdYlGn_r', alpha=0.4, s=8)
plt.colorbar(sc, ax=ax, label='Is Fraud')
ax.set_title('Credit Score vs Log(Amount)', color='#e6edf3')
ax.set_xlabel('Credit Score'); ax.set_ylabel('log(Amount)')

# 2d. Channel heatmap by time of day
pivot_ch = df.groupby(['channel','time_of_day'])['transaction_id'].count().unstack(fill_value=0)
pivot_ch = pivot_ch[['Morning','Afternoon','Evening','Night']]
ax = axes[1,0]
sns.heatmap(pivot_ch, annot=True, fmt=',d', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label':'Count'}, annot_kws={'size':8})
ax.set_title('Channel × Time of Day Heatmap', color='#e6edf3')
ax.set_xlabel('Time of Day'); ax.set_ylabel('Channel')
ax.tick_params(colors='#e6edf3')

# 2e. Amount category vs transaction status
pivot_as = df.groupby(['amount_category','transaction_status'])['transaction_id'].count().unstack(fill_value=0)
cat_order = ['Micro (<500)','Small (500-5K)','Medium (5K-50K)','Large (50K-5L)','Very Large (>5L)']
pivot_as = pivot_as.reindex([c for c in cat_order if c in pivot_as.index])
pivot_as_pct = pivot_as.div(pivot_as.sum(axis=1), axis=0) * 100
pivot_as_pct.plot(kind='bar', ax=axes[1,1], color=ACCENT[:4], edgecolor='none', stacked=True)
axes[1,1].set_title('Amount Category × Status (%)', color='#e6edf3')
axes[1,1].set_xlabel('Amount Category')
axes[1,1].set_ylabel('% of Transactions')
axes[1,1].tick_params(axis='x', rotation=35)
axes[1,1].legend(fontsize=8)

# 2f. Correlation heatmap
ax = axes[1,2]
corr_cols = ['transaction_amount','account_balance','credit_score','emi_amount',
             'has_loan','is_fraud','transaction_hour','is_weekend','is_high_value']
corr = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', ax=ax,
            vmin=-1, vmax=1, linewidths=0.5, annot_kws={'size':7})
ax.set_title('Correlation Matrix', color='#e6edf3')
ax.tick_params(axis='x', rotation=45, colors='#e6edf3')
ax.tick_params(axis='y', colors='#e6edf3')

plt.tight_layout()
plt.savefig('/content/sample_data/fig2_multivariate.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.close()
print("[FIG 2] Saved: fig2_multivariate.png")

# ─── SAVE SQL-STYLE RESULTS ──────────────────────────────────
results = {
    'Q1_top_merchants': q1.reset_index(),
    'Q2_monthly_trend': q2,
    'Q3_fraud_by_state': q3.reset_index(),
    'Q4_success_by_channel': q4.reset_index(name='success_rate_pct'),
    'Q5_avg_amount': q5.reset_index(),
    'Q6_kyc_fraud': q6.reset_index(),
    'Q7_state_volume': q7.reset_index(name='total_volume')
}
with pd.ExcelWriter('/content/sample_data/eda_sql_results.xlsx', engine='openpyxl') as writer:
    for sheet, data in results.items():
        data.to_excel(writer, sheet_name=sheet[:31], index=False)
print("[SAVED] eda_sql_results.xlsx")
print("\n✅ Task 2 Complete!")

TASK 2: Exploratory Data Analysis & Business Intelligence
Loaded: 550,000 rows × 32 columns

--- STEP 1: DESCRIPTIVE STATISTICS ---

Numerical Summary:
       transaction_amount  account_balance  credit_score  emi_amount
count           550000.00        550000.00     550000.00   550000.00
mean             29907.09         84550.12        599.81     2365.86
std             139610.10        170914.59        173.05     4973.55
min                  2.40           500.00        300.00        0.00
25%                763.04         15157.78        450.00        0.00
50%               2035.74         36422.87        600.00        0.00
75%               8557.50         87607.04        750.00     3115.03
max           10000000.00       5000000.00        899.00   153325.74

Transaction Direction Split:
transaction_direction
Debit     67.34
Credit    32.66

Transaction Status Split:
transaction_status
Success     92.04
Failed       3.98
Reversed     2.00
Pending      1.97

Fraud Rate: 0.886%
[FIG 